## Day 5 Reflection

What I learned:

- A vector database stores vectors along with documents and metadata.
- Chroma can perform similarity search over stored embeddings.
- Smaller distance means a retrieved vector is closer according to the configured metric.
- Metadata filters can combine semantic search with structured constraints.
- Persistent vector databases allow data to survive application/runtime restarts.
- Exact nearest-neighbor search compares against all vectors, while ANN methods such as HNSW trade some recall for better search performance.
- HNSW search parameters such as ef_search affect the recall-latency trade-off.
- Embedding models create vector representations, while vector databases store and retrieve them.
- A vector database does not generate answers; it retrieves relevant information.
- LangChain provides abstractions that make it easier to connect embeddings, vector stores, retrievers, prompts, and LLMs.

Key takeaway:

A vector database is the retrieval layer of an AI application.
It finds relevant information; the LLM uses that information to
generate the final response.

In [2]:
!pip -q install chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 116.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

Create our Embedding Model

In [8]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
documents = [
    "Python generators produce values lazily.",
    "FastAPI is a framework for building APIs.",
    "Machine learning models learn patterns from data.",
    "Python lists store collections of items.",
    "Chocolate cake is made with cocoa and flour."
]

In [5]:
embeddings = model.encode(documents)

print("Number of documents:", len(documents))
print("Embedding shape:", embeddings.shape)

Number of documents: 5
Embedding shape: (5, 384)


Create a Chroma Collection

In [6]:
import chromadb

client = chromadb.Client()
collection = client.create_collection(
    name="ai_engineering_day5"
)

Store our documents

In [7]:
collection.add(
    ids=[f"doc_{i}" for i in range (len(documents))],
    documents=documents,
    embeddings=embeddings.tolist()
)

print(collection.count())

5


Search it

In [8]:
query = "How can Python produce values one at a time?"

query_embedding = model.encode([query])[0]

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3
)

print(results["documents"])

[['Python generators produce values lazily.', 'Python lists store collections of items.', 'Machine learning models learn patterns from data.']]


Let's inspect the similarity scores

In [9]:
results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3,
    include=["documents", "distances"]
)

print(results)

{'ids': [['doc_0', 'doc_3', 'doc_2']], 'embeddings': None, 'documents': [['Python generators produce values lazily.', 'Python lists store collections of items.', 'Machine learning models learn patterns from data.']], 'uris': None, 'included': ['documents', 'distances'], 'data': None, 'metadatas': None, 'distances': [[0.7947283983230591, 1.0576545000076294, 1.607656478881836]]}


Change n_results

In [13]:
results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5,
    include=["documents", "distances"]
)

for doc, distance in zip(
    results["documents"][0], results["distances"][0]
):
  print(f"Distance: {distance:.4f}")
  print(doc)
  print()


Distance: 0.7947
Python generators produce values lazily.

Distance: 1.0577
Python lists store collections of items.

Distance: 1.6077
Machine learning models learn patterns from data.

Distance: 1.9010
FastAPI is a framework for building APIs.

Distance: 2.1286
Chocolate cake is made with cocoa and flour.



Exact Nearest Neighbor vs ANN

Exact Nearest Neighbor (Exact k-NN) performs an exhaustive, brute-force linear scan comparing a query vector against every single vector in a dataset, guaranteeing 100% accuracy at the expense of linear time complexity (O(N)), whereas Approximate Nearest Neighbor (ANN) builds specialized indexes (such as HNSW or IVF) to trade a minor, configurable drop in recall for sublinear, sub-second query times at scale

  Add Metadata to our Chroma Collection

In [14]:
collection = client.create_collection(
    name="ai_engineering_matadata"
)

In [15]:
documents=[
    "Python generators produce values lazily.",
    "FastAPI is a framework for building APIs.",
    "Machine learning models learn patterns from data.",
    "Python lists store collections of items."
]

In [16]:
embeddings = model.encode(documents)

In [17]:
metadata = [
    {
        "topic": "python",
        "type": "language"
    },
    {
        "topic": "fastapi",
        "type": "framework"
    },
    {
        "topic": "machine learning",
        "type": "ai"
    },
    {
        "topic": "python",
        "type": "language"
    }
]

In [19]:
collection.add(
    ids=[f"doc_{i}" for i in range(len(documents))],
    documents=documents,
    embeddings=embeddings.tolist(),
    metadatas=metadata
)

Now search normally

In [22]:
query = "How does Python generate values lazily?"

query_embedding = model.encode([query])[0]

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

print(results)

{'ids': [['doc_0', 'doc_3', 'doc_2']], 'embeddings': None, 'documents': [['Python generators produce values lazily.', 'Python lists store collections of items.', 'Machine learning models learn patterns from data.']], 'uris': None, 'included': ['documents', 'metadatas', 'distances'], 'data': None, 'metadatas': [[{'type': 'language', 'topic': 'python'}, {'topic': 'python', 'type': 'language'}, {'type': 'ai', 'topic': 'machine learning'}]], 'distances': [[0.28299835324287415, 1.2673311233520508, 1.6086461544036865]]}


Search only Python documents

In [24]:
results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3,
    include=["documents", "metadatas", "distances"],
    where={"topic": "python"}
)

print(results)

{'ids': [['doc_0', 'doc_3']], 'embeddings': None, 'documents': [['Python generators produce values lazily.', 'Python lists store collections of items.']], 'uris': None, 'included': ['documents', 'metadatas', 'distances'], 'data': None, 'metadatas': [[{'type': 'language', 'topic': 'python'}, {'topic': 'python', 'type': 'language'}]], 'distances': [[0.28299835324287415, 1.2673311233520508]]}


Persistent Chroma

In [4]:
import chromadb

client = chromadb.PersistentClient(
    path="./chroma_db"
)

In [5]:
collection = client.get_or_create_collection(
    name="python_docs"
)

In [6]:
documents = [
    "Python generators produce values lazily.",
    "FastAPI is a framework for building APIs.",
    "Machine learning models learn patterns from data.",
    "Python lists store collections of items."
]

In [9]:
embeddings = model.encode(documents)

In [12]:
collection.add(
    ids=[f"doc_{i}" for i in range(len(documents))],
    documents=documents,
    embeddings=embeddings.tolist()
)

print(collection.count())

4


Experiment 1 — Let's inspect Chroma's default configuration

In [13]:
collection = client.get_collection("python_docs")

print(collection.configuration_json)

{'hnsw': {'space': 'l2', 'ef_construction': 100, 'ef_search': 100, 'max_neighbors': 16, 'resize_factor': 1.2, 'sync_threshold': 1000}, 'spann': None, 'embedding_function': {'type': 'known', 'name': 'default', 'config': {}}}


In [14]:
print(collection.metadata)

None


Experiment 2 — Explicitly choose the distance metric



*   Cosine -> compares direction

*   L2/Euclidean -> compares geometric distance
*   Inner Product -> compares vector alignment/magnitide





In [15]:
cosine_collection = client.get_or_create_collection(
    name="python_cosine",
    configuration={
        "hnsw":{
            "space": "cosine"
        }
    }
)

In [16]:
l2_collection = client.get_or_create_collection(
    name="python_l2",
    configuration={
        "hnsw":{
            "space": "l2"
        }
    }
)

In [17]:
ip_collection = client.get_or_create_collection(
    name="python_ip",
    configuration={
        "hnsw":{
            "space": "ip"
        }
    }
)

Experiment 3 — Put the same vectors into all three

In [20]:
embeddings = model.encode(documents)

ids = [f"doc_{i}" for i in range(len(documents))]

cosine_collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings.tolist()
)

l2_collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings.tolist()
)

ip_collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings.tolist()
)

In [21]:
query = "How does Python generate values lazily?"

query_embedding = model.encode([query])[0]

In [22]:
cosine_results = cosine_collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=4,
    include=["documents", "distances"]
)

In [23]:
l2_results = l2_collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=4,
    include=["documents", "distances"]
)

In [24]:
ip_results = ip_collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=4,
    include=["documents", "distances"]
)

In [25]:
print("COSINE")
for doc, distance in zip(
    cosine_results["documents"][0],
    cosine_results["distances"][0]
):
    print(distance, "->", doc)

print("\nL2")
for doc, distance in zip(
    l2_results["documents"][0],
    l2_results["distances"][0]
):
    print(distance, "->", doc)

print("\nINNER PRODUCT")
for doc, distance in zip(
    ip_results["documents"][0],
    ip_results["distances"][0]
):
    print(distance, "->", doc)

COSINE
0.14149856567382812 -> Python generators produce values lazily.
0.6336655616760254 -> Python lists store collections of items.
0.8043230772018433 -> Machine learning models learn patterns from data.
0.8849191665649414 -> FastAPI is a framework for building APIs.

L2
0.28299835324287415 -> Python generators produce values lazily.
1.2673311233520508 -> Python lists store collections of items.
1.6086461544036865 -> Machine learning models learn patterns from data.
1.7698383331298828 -> FastAPI is a framework for building APIs.

INNER PRODUCT
0.1414991021156311 -> Python generators produce values lazily.
0.6336656212806702 -> Python lists store collections of items.
0.8043230772018433 -> Machine learning models learn patterns from data.
0.8849192261695862 -> FastAPI is a framework for building APIs.


In [28]:
import numpy as np

A = np.array([1.0, 0.0])
B = np.array([10.0, 0.0])
Q = np.array([1.0, 0.0])

In [29]:
from sklearn.metrics.pairwise import cosine_similarity

print("Q vs A:",
      cosine_similarity([Q], [A])[0][0])

print("Q vs B:",
      cosine_similarity([Q], [B])[0][0])



Q vs A: 1.0
Q vs B: 1.0


In [30]:
from sklearn.metrics import euclidean_distances

print("Q vs A:",
      euclidean_distances([Q], [A])[0][0])

print("Q vs B:",
      euclidean_distances([Q], [B])[0][0])

Q vs A: 0.0
Q vs B: 9.0


HNSW: Recall vs Search Effort

Step 1 — Create a larger dataset

In [31]:
import numpy as np

np.random.seed(42)

num_vectors = 10_000
dimension = 128

data = np.random.randn(
    num_vectors,
    dimension
).astype("float32")

print(data.shape)

(10000, 128)


Step 2 — Create a query

In [32]:
query = np.random.randn(
    1,
    dimension
).astype("float32")

Step 3 — Establish the ground truth

In [33]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(
    query,
    data
)[0]

In [34]:
ground_truth = np.argsort(
    similarities
)[::-1][:10]

print("True top-10 indices:")
print(ground_truth)

True top-10 indices:
[4921 8572 3707 6667 9886 3416 8681 5733 1302 8149]


Step 5 — Create an HNSW index

In [35]:
!pip -q install hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [36]:
import hnswlib

index = hnswlib.Index(
    space="cosine",
    dim=dimension
)

index.init_index(
    max_elements=num_vectors,
    ef_construction=200,
    M=16
)

In [38]:
index.add_items(
    data,
    np.arange(num_vectors)
)

M controls how many connections a node can maintain in the HNSW graph


m - small - weak navigation

m - larger - better connectivity

ef_construction - This controls how much effort is spent building the HNSW index.

In [39]:
index.set_ef(10)

labels, distance = index.knn_query(
    query,
    k=10
)

ann_results = labels[0]

print("ANN results:")
print(ann_results)

ANN results:
[4921 8572 6667 5033  332  321  203 3260 2689 2000]


Step 9 — Calculate Recall@10

In [40]:
def recall_at_k(ground_truth, retrived):
  ground_truth = set(ground_truth)
  retrived = set(retrived)

  return len(ground_truth.intersection(retrived))/ len(ground_truth)

In [41]:
recall = recall_at_k(
    ground_truth,
    ann_results
)

print("Recall@10:", recall)

Recall@10: 0.3


Step 10 — Now increase ef

In [42]:
for ef in [10,20,50,100,200]:
  index.set_ef(ef)

  labels, distances = index.knn_query(
      query,
      k=10
  )

  retrived = labels[0]

  recall = recall_at_k(
      ground_truth,
      retrived
  )

  print(
      f"Recall@10 with ef={ef}:",
      recall
  )

Recall@10 with ef=10: 0.3
Recall@10 with ef=20: 0.5
Recall@10 with ef=50: 0.8
Recall@10 with ef=100: 0.9
Recall@10 with ef=200: 1.0


Measure latency

In [43]:
import time

for ef in [10, 20, 50, 100, 200]:

    index.set_ef(ef)

    start = time.perf_counter()

    labels, distances = index.knn_query(
        query,
        k=10
    )

    end = time.perf_counter()

    retrieved = labels[0]

    recall = recall_at_k(
        ground_truth,
        retrieved
    )

    latency_ms = (end - start) * 1000

    print(
        f"ef={ef:3d} | "
        f"Recall@10={recall:.2f} | "
        f"Latency={latency_ms:.4f} ms"
    )

ef= 10 | Recall@10=0.30 | Latency=6.2723 ms
ef= 20 | Recall@10=0.50 | Latency=0.1924 ms
ef= 50 | Recall@10=0.80 | Latency=0.2966 ms
ef=100 | Recall@10=0.90 | Latency=0.3835 ms
ef=200 | Recall@10=1.00 | Latency=8.7513 ms


Final Build: Semantic Search with Chroma

In [44]:
documents = [
    "Python generators produce values lazily. They allow programs to generate values one at a time instead of storing the entire sequence in memory.",

    "Python lists are mutable sequences that can store multiple values. Lists keep their elements in order and support indexing and slicing.",

    "FastAPI is a modern Python web framework for building APIs. It provides automatic request validation and interactive API documentation.",

    "FastAPI uses Python type hints to validate request data and generate OpenAPI documentation automatically.",

    "Machine learning models learn patterns from data and use those patterns to make predictions or decisions on new inputs.",

    "Vector databases store numerical vector representations of data and allow applications to search for similar vectors efficiently.",

    "Embeddings convert text into numerical vectors that capture semantic information about the text.",

    "Chroma is a vector database that can store documents, embeddings, IDs, and metadata and retrieve similar documents.",

    "Docker packages applications and their dependencies into containers so that they can run consistently across different environments.",

    "Redis is an in-memory data store commonly used for caching, queues, sessions, and fast temporary data access."
]

Create embeddings

In [45]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = model.encode(documents)

print("Number of documents:", len(documents))
print("Embedding dimensions:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Number of documents: 10
Embedding dimensions: (10, 384)


Create a persistent Chroma database

In [46]:
import chromadb

client = chromadb.PersistentClient(
    path="./chroma_day5"
)

In [47]:
collection = client.get_or_create_collection(
    name="ai_engineering_docs"
)

Add documents + vectors + metadata

In [48]:
metadata = [
    {"topic": "python", "source": "python_docs"},
    {"topic": "python", "source": "python_docs"},
    {"topic": "fastapi", "source": "fastapi_docs"},
    {"topic": "fastapi", "source": "fastapi_docs"},
    {"topic": "machine_learning", "source": "ml_docs"},
    {"topic": "vector_database", "source": "vector_docs"},
    {"topic": "embeddings", "source": "embedding_docs"},
    {"topic": "vector_database", "source": "chroma_docs"},
    {"topic": "docker", "source": "docker_docs"},
    {"topic": "redis", "source": "redis_docs"},
]

In [49]:
collection.add(
    ids=[f"doc_{i}" for i in range(len(documents))],
    documents=documents,
    embeddings=embeddings.tolist(),
    metadatas=metadata
)

In [50]:
print("Documents stored:", collection.count())

Documents stored: 10


Search it

In [51]:
query = "How can I generate values one at a time in Python?"

query_embedding = model.encode([query])[0]

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=2,
    include=["documents", "metadatas", "distances"]
)

In [52]:
for i, (doc, metadata, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\nResult {i}")
    print("Distance:", round(distance, 4))
    print("Metadata:", metadata)
    print("Document:", doc)


Result 1
Distance: 0.7679
Metadata: {'source': 'python_docs', 'topic': 'python'}
Document: Python generators produce values lazily. They allow programs to generate values one at a time instead of storing the entire sequence in memory.

Result 2
Distance: 1.0108
Metadata: {'topic': 'python', 'source': 'python_docs'}
Document: Python lists are mutable sequences that can store multiple values. Lists keep their elements in order and support indexing and slicing.


Add metadata filtering

In [53]:
results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3,
    where={"topic": "python"},
    include=[
        "documents",
        "metadatas",
        "distances"
    ]
)

Test persistence

In [54]:
new_client = chromadb.PersistentClient(
    path="./chroma_day5"
)

new_collection = new_client.get_collection(
    name="ai_engineering_docs"
)

print("Documents after reopening:",
      new_collection.count())

Documents after reopening: 10
